Assignment 07 - Deep Learning for NLP

Look into twitter text data to predict if the given text has positive or negative
sentiment towards a particular brand. The dataset includes twitter text related to
Apple and Google products with user sentiment ranked between ‘positive’,
‘negative’, ‘neutral’ and ‘no_idea’, sentiments. Create a simpleRNN or LSTM based
classifiers to classify tweets into the four classes. You can avoid
‘emotion_in_tweet_is_directed_at’ column.


Load Data

In [45]:
import pandas as pd
df = pd.read_csv('/content/tweets2.csv', encoding='ISO-8859-1')
# drop ‘emotion_in_tweet_is_directed_at’ column.
df = df[['tweet_text', 'is_there_an_emotion_directed_at_a_brand_or_product']]
df.head(5)


,tweet_text,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Positive emotion


In [46]:
# rename columns
df.columns = ['text', 'sentiment']
df.head(5)

,text,sentiment
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Positive emotion


In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9093 entries, 0 to 9092
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       9092 non-null   object
 1   sentiment  9093 non-null   object
dtypes: object(2)
memory usage: 142.2+ KB


In [48]:
df.describe()

,text,sentiment
count,9092,9093
unique,9065,4
top,RT @mention Marissa Mayer: Google Will Connect...,No emotion toward brand or product
freq,5,5389


Clean Data

In [49]:
# Find Missing
df.isna().mean()

,0
text,0.00011
sentiment,0.00000


In [50]:
# drop the row
df = df.dropna(subset=['text'])

In [51]:
df.isna().mean()  # missing value handled

,0
text,0.0
sentiment,0.0


In [52]:
import numpy as np
import re
import string

In [53]:
# Cleaning text data by converting to lowercase and removing URLs, user mentions, punctuation, and numbers to reduce noise for the model.
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    return text.strip()

df['text'] = df['text'].apply(clean_text)


In [54]:
#  Encode Labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['label'] = le.fit_transform(df['sentiment']) # label - 4 classes
num_classes = len(le.classes_)

In [55]:
#  Tokenization & Padding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_words = 5000
max_len = 50
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['text'])

Text and Target

In [56]:
X = pad_sequences(tokenizer.texts_to_sequences(df['text']), maxlen=max_len)
y = df['label'].values

In [57]:
print(df['label'].value_counts()) # imbalanced classes

label
2    5388
3    2978
1     570
0     156
Name: count, dtype: int64


Stratified Split

In [58]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) # for imbalanced data

Balancing with SMOTE

used SMOTE to physically create synthetic examples for minority classes, allowing the LSTM to learn their specific patterns more effectively than simply weighting existing samples. As weighing classes only led them predicting the majority classes

In [59]:
#  Apply SMOTE to training data only
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"New class distribution: {np.bincount(y_train_smote)}")

New class distribution: [4310 4310 4310 4310]


Build Model

In [60]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D
model = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    SpatialDropout1D(0.3),
    LSTM(100, dropout=0.2, recurrent_dropout=0.2),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Train the model

Stopped at epoch 3 as it is where the model achieved its lowest Validation Loss before it began to overfit.

In [61]:
model.fit(X_train_smote, y_train_smote, epochs=3, batch_size=64, validation_data=(X_test, y_test))

Epoch 1/3
270/270 ━━━━━━━━━━━━━━━━━━━━ 42s 139ms/step - accuracy: 0.3868 - loss: 1.2660 - val_accuracy: 0.5327 - val_loss: 1.0777
Epoch 2/3
270/270 ━━━━━━━━━━━━━━━━━━━━ 39s 131ms/step - accuracy: 0.4824 - loss: 1.1273 - val_accuracy: 0.5569 - val_loss: 1.0306
Epoch 3/3
270/270 ━━━━━━━━━━━━━━━━━━━━ 41s 130ms/step - accuracy: 0.5377 - loss: 1.0291 - val_accuracy: 0.5591 - val_loss: 1.0635


Evaluation

In [62]:
# ACCURACY
from sklearn.metrics import accuracy_score
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

Accuracy: 54.81%


In [63]:
# CLASSIFICATION REPORT
from sklearn.metrics import classification_report
y_pred = np.argmax(model.predict(X_test), axis=1)
print("\n### Classification Report ###")
print(classification_report(y_test, y_pred, target_names=le.classes_))

57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step

### Classification Report ###
                                    precision    recall  f1-score   support

                      I can't tell       0.02      0.06      0.03        31
                  Negative emotion       0.19      0.30      0.23       114
No emotion toward brand or product       0.71      0.66      0.68      1078
                  Positive emotion       0.52      0.45      0.49       596

                          accuracy                           0.56      1819
                         macro avg       0.36      0.37      0.36      1819
                      weighted avg       0.60      0.56      0.58      1819



The LSTM-based sentiment classifier achieved moderate performance with approx 55% accuracy. While it effectively learned the majority class, it struggled with minority classes due to dataset imbalance, resulting in low recall and F1-scores for underrepresented sentiments.